<a href="https://colab.research.google.com/github/yoyers/Grain_size_analysis/blob/main/notebooks/grain_size_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys

if not os.path.exists("/content/Grain_size_analysis"):
    !git clone https://github.com/yoyers/Grain_size_analysis.git

%cd /content/Grain_size_analysis
!git pull
!pip install -q cellpose

sys.path.append("/content/Grain_size_analysis")

import torch
print("GPU available:", torch.cuda.is_available())

/content/Grain_size_analysis
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 535 bytes | 178.00 KiB/s, done.
From https://github.com/yoyers/Grain_size_analysis
   dcc8c92..c98f208  main       -> origin/main
Updating dcc8c92..c98f208
Fast-forward
 grain_size/analysis.py | 6 ++++++
 1 file changed, 6 insertions(+)
GPU available: True


In [2]:
#@title Settings
from google.colab import drive
drive.mount("/content/drive")

data_folder = "/content/drive/MyDrive/Data Science/PhD Experiments/Grain_size_analysis/data/Test"  #@param {type:"string"}
results_folder = "/content/drive/MyDrive/Data Science/PhD Experiments/Grain_size_analysis/results/Test"  #@param {type:"string"}
nm_per_pixel = 22.33  #@param {type:"number"}
crop_row = 708  #@param {type:"integer"}
min_grain_nm = 100  #@param {type:"number"}
flow_threshold = 0.4  #@param {type:"number"}
cellprob_threshold = 0.0  #@param {type:"number"}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

from grain_size.analysis import (load_image, load_model, segment_cellpose,
                                 measure_grains)
from grain_size.plots import save_results

extensions = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}
files = sorted(f for f in Path(data_folder).iterdir()
               if f.suffix.lower() in extensions)
print(len(files), "images found")

results = Path(results_folder)
results.mkdir(parents=True, exist_ok=True)

model = load_model(gpu=True)          # loaded once for all images

summary = []
all_grains = []

for i, path in enumerate(files, start=1):
    stem = path.stem
    print(f"[{i}/{len(files)}] {path.name}")
    try:
        img = load_image(path, crop_row=crop_row)
        masks = segment_cellpose(img, flow_threshold=flow_threshold,
                                 cellprob_threshold=cellprob_threshold,
                                 model=model)
        labels, df = measure_grains(masks, nm_per_pixel=nm_per_pixel,
                                    min_grain_nm=min_grain_nm)
        np.save(results / f"{stem}_masks.npy", masks)
        save_results(img, labels, df, results, stem)
    except Exception as e:
        print("   FAILED:", e)
        continue

    df.insert(0, "image", stem)       # after saving the per-image CSV
    all_grains.append(df)
    summary.append({
        "image": stem,
        "n_grains": len(df),
        "mean_diameter_nm": df["diameter_nm"].mean(),
        "median_diameter_nm": df["diameter_nm"].median(),
        "std_diameter_nm": df["diameter_nm"].std(),
    })
    print(f"   {len(df)} grains | mean {df['diameter_nm'].mean():.0f} nm")

if summary:
    pd.DataFrame(summary).to_csv(results / "summary_all_images.csv", index=False)
    pd.concat(all_grains).to_csv(results / "grains_all_images.csv", index=False)
    print("\nDone. Summary saved in", results)
else:
    print("No image was processed.")

1 images found


100%|██████████| 1.15G/1.15G [00:13<00:00, 94.2MB/s]


[1/1] Test-30092026-400bars-300C_06.tif


/usr/local/lib/python3.13/dist-packages/cellpose/dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


   394 grains | mean 889 nm

Done. Summary saved in /content/drive/MyDrive/Data Science/PhD Experiments/Grain_size_analysis/results/Test
